# 11 - Silver Orders

Builds latest-state Orders with `MERGE INTO`, FX conversion, validation, and an append-like status history maintained by MERGE.

In [ ]:
%run ./05_bronze_common

In [ ]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


def run_version(run_id: str) -> int:
    return int(run_id.replace("-", "")[:15], 16)


def merge_quarantine(table_name: str, errors) -> int:
    count = errors.count()
    if count:
        (
            DeltaTable.forName(spark, table_name)
            .alias("target")
            .merge(errors.alias("source"), "target.error_id = source.error_id")
            .whenNotMatchedInsertAll()
            .execute()
        )
    return count


def silver_dq(
    catalog: str,
    source_name: str,
    pipeline_run_id: str,
    valid_count: int,
    errors,
    duplicate_count: int = 0,
) -> None:
    error_counts = errors.groupBy("error_code").count().collect()
    if duplicate_count:
        error_counts.append(Row(error_code="duplicate_key", count=duplicate_count))
    now = datetime.now(timezone.utc)
    rows = [
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="silver",
            quality_status="valid",
            error_code=None,
            record_count=int(valid_count),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
    ]
    rows.extend(
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="silver",
            quality_status="error",
            error_code=row.error_code,
            record_count=int(row["count"]),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
        for row in error_counts
    )
    append_delta(
        spark.createDataFrame(rows, DQ_SCHEMA),
        f"{catalog}.control.data_quality_results",
        f"{source_name}_silver_dq",
        run_version(pipeline_run_id),
    )


def silver_error(
    error_code,
    error_reason,
    record_key,
    raw_payload,
    rescued_data,
    source_file,
    batch_id,
    load_date,
    ingest_ts,
    pipeline_run_id,
):
    return [
        F.sha2(
            F.concat_ws(
                "||",
                source_file,
                F.coalesce(record_key, F.lit("")),
                raw_payload,
                error_code,
            ),
            256,
        ).alias("error_id"),
        F.lit("silver").alias("error_stage"),
        error_code.alias("error_code"),
        error_reason.alias("error_reason"),
        record_key.alias("record_key"),
        raw_payload.alias("raw_payload"),
        rescued_data.alias("rescued_data"),
        source_file.alias("source_file"),
        batch_id.alias("batch_id"),
        load_date.alias("load_date"),
        ingest_ts.alias("ingest_ts"),
        F.current_date().alias("quarantine_date"),
        pipeline_run_id.alias("pipeline_run_id"),
    ]

In [ ]:
dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")
catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`silver`")

pipeline_name = "silver_orders"
source_name = "orders"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
bronze_table = f"{catalog}.bronze.bronze_orders"
target_table = f"{catalog}.silver.silver_orders"
history_table = f"{catalog}.silver.silver_order_status_history"
quarantine_table = f"{catalog}.quarantine.orders_errors"

item_schema = T.ArrayType(T.StructType([
    T.StructField("sku", T.StringType(), True),
    T.StructField("qty", T.LongType(), True),
    T.StructField("unit_price", T.DecimalType(18, 2), True),
]))
address_schema = T.StructType([
    T.StructField("line1", T.StringType(), True),
    T.StructField("city", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("postal_code", T.StringType(), True),
])

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`silver`.`silver_orders` (
        order_id STRING NOT NULL,
        order_time TIMESTAMP NOT NULL,
        order_date DATE NOT NULL,
        user_id STRING,
        items ARRAY<STRUCT<sku: STRING, qty: BIGINT, unit_price: DECIMAL(18,2)>>,
        currency STRING NOT NULL,
        discount_code STRING,
        shipping_address STRUCT<line1: STRING, city: STRING, country: STRING, postal_code: STRING>,
        status STRING NOT NULL,
        updated_at TIMESTAMP NOT NULL,
        fx_rate_to_usd DECIMAL(20,8) NOT NULL,
        subtotal_original DECIMAL(24,2) NOT NULL,
        subtotal_usd DECIMAL(24,2) NOT NULL,
        ingest_ts TIMESTAMP NOT NULL,
        source_file STRING NOT NULL,
        batch_id STRING NOT NULL,
        load_date DATE NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    """
)
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`silver`.`silver_order_status_history` (
        order_id STRING NOT NULL,
        status STRING NOT NULL,
        status_ts TIMESTAMP NOT NULL,
        order_time TIMESTAMP NOT NULL,
        user_id STRING,
        country STRING,
        subtotal_usd DECIMAL(24,2) NOT NULL,
        ingest_ts TIMESTAMP NOT NULL,
        source_file STRING NOT NULL,
        batch_id STRING NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    """
)

upsert_audit(catalog, pipeline_run_id, pipeline_name, "full_bronze_merge", start_ts, "RUNNING", [bronze_table], job_run_id)
try:
    bronze = spark.table(bronze_table)
    parsed = (
        bronze
        .withColumn("order_id_parsed", F.get_json_object("raw_payload", "$.order_id"))
        .withColumn("order_time_parsed", F.expr("try_to_timestamp(get_json_object(raw_payload, '$.order_time'))"))
        .withColumn("updated_at_parsed", F.expr("try_to_timestamp(get_json_object(raw_payload, '$.updated_at'))"))
        .withColumn("user_id_parsed", F.get_json_object("raw_payload", "$.user_id"))
        .withColumn("currency_parsed", F.upper(F.get_json_object("raw_payload", "$.currency")))
        .withColumn("discount_code_parsed", F.get_json_object("raw_payload", "$.discount_code"))
        .withColumn("status_parsed", F.get_json_object("raw_payload", "$.status"))
        .withColumn("items_parsed", F.from_json(F.get_json_object("raw_payload", "$.items"), item_schema))
        .withColumn("address_parsed", F.from_json(F.get_json_object("raw_payload", "$.shipping_address"), address_schema))
        .withColumn(
            "invalid_items",
            F.expr("items_parsed is null or size(items_parsed) = 0 or exists(items_parsed, x -> x.sku is null or x.qty <= 0 or x.unit_price < 0)"),
        )
        .withColumn("subtotal_original", F.expr("aggregate(items_parsed, cast(0 as decimal(24,2)), (acc, x) -> cast(acc + cast(x.qty * x.unit_price as decimal(24,2)) as decimal(24,2)))"))
        .join(
            spark.table(f"{catalog}.silver.dim_fx_rates").select("rate_date", "currency", "rate_to_usd"),
            (F.to_date("order_time_parsed") == F.col("rate_date"))
            & (F.col("currency_parsed") == F.col("currency")),
            "left",
        )
        .withColumn("subtotal_usd", (F.col("subtotal_original") * F.col("rate_to_usd")).cast("decimal(24,2)"))
        .withColumn(
            "error_code",
            F.when(F.col("order_id_parsed").isNull(), F.lit("missing_key"))
            .when(F.col("order_time_parsed").isNull() | F.col("updated_at_parsed").isNull(), F.lit("invalid_timestamp"))
            .when(F.col("invalid_items"), F.lit("schema_mismatch"))
            .when(F.col("rate_to_usd").isNull(), F.lit("fx_rate_missing")),
        )
        .withColumn(
            "error_reason",
            F.when(F.col("error_code") == "missing_key", F.lit("order_id is null"))
            .when(F.col("error_code") == "invalid_timestamp", F.lit("order_time or updated_at cannot be parsed"))
            .when(F.col("error_code") == "schema_mismatch", F.lit("items must be non-empty with positive qty and non-negative price"))
            .when(F.col("error_code") == "fx_rate_missing", F.lit("No FX rate for order_date and currency")),
        )
    )
    errors = parsed.filter(F.col("error_code").isNotNull()).select(
        *silver_error(
            F.col("error_code"), F.col("error_reason"), F.col("order_id_parsed"),
            F.col("raw_payload"), F.lit(None).cast("string"), F.col("source_file"),
            F.col("batch_id"), F.col("load_date"), F.col("ingest_ts"),
            F.lit(pipeline_run_id),
        )
    )
    error_count = merge_quarantine(quarantine_table, errors)
    valid = parsed.filter(F.col("error_code").isNull()).select(
        F.col("order_id_parsed").alias("order_id"),
        F.col("order_time_parsed").alias("order_time"),
        F.to_date("order_time_parsed").alias("order_date"),
        F.col("user_id_parsed").alias("user_id"),
        F.col("items_parsed").alias("items"),
        F.col("currency_parsed").alias("currency"),
        F.col("discount_code_parsed").alias("discount_code"),
        F.col("address_parsed").alias("shipping_address"),
        F.col("status_parsed").alias("status"),
        F.col("updated_at_parsed").alias("updated_at"),
        F.col("rate_to_usd").cast("decimal(20,8)").alias("fx_rate_to_usd"),
        F.col("subtotal_original").cast("decimal(24,2)").alias("subtotal_original"),
        F.col("subtotal_usd").cast("decimal(24,2)").alias("subtotal_usd"),
        "ingest_ts", "source_file", "batch_id", "load_date",
        F.lit(pipeline_run_id).alias("pipeline_run_id"),
    )

    history = valid.select(
        "order_id", "status", F.col("updated_at").alias("status_ts"), "order_time",
        "user_id", F.col("shipping_address.country").alias("country"), "subtotal_usd",
        "ingest_ts", "source_file", "batch_id", "pipeline_run_id",
    ).dropDuplicates(["order_id", "status_ts"])
    (
        DeltaTable.forName(spark, history_table).alias("target")
        .merge(history.alias("source"), "target.order_id = source.order_id AND target.status_ts = source.status_ts")
        .whenMatchedUpdateAll(condition="source.ingest_ts > target.ingest_ts")
        .whenNotMatchedInsertAll().execute()
    )

    ranking = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc(), F.col("ingest_ts").desc(), F.col("source_file").desc())
    latest = valid.withColumn("_rank", F.row_number().over(ranking)).filter(F.col("_rank") == 1).drop("_rank")
    (
        DeltaTable.forName(spark, target_table).alias("target")
        .merge(latest.alias("source"), "target.order_id = source.order_id")
        .whenMatchedUpdate(condition="source.updated_at > target.updated_at OR (source.updated_at = target.updated_at AND source.ingest_ts > target.ingest_ts) OR (source.updated_at = target.updated_at AND source.ingest_ts = target.ingest_ts AND source.source_file > target.source_file)", set={column: f"source.{column}" for column in latest.columns})
        .whenNotMatchedInsertAll().execute()
    )
    valid_count = latest.count()
    silver_dq(catalog, source_name, pipeline_run_id, valid_count, errors)
    upsert_audit(catalog, pipeline_run_id, pipeline_name, "full_bronze_merge", start_ts, "SUCCESS", [bronze_table], job_run_id, end_ts=datetime.now(timezone.utc), rows_read=bronze.count(), rows_written=valid_count, rows_quarantined=error_count)
except Exception as exc:
    upsert_audit(catalog, pipeline_run_id, pipeline_name, "full_bronze_merge", start_ts, "FAILED", [bronze_table], job_run_id, end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000])
    raise